In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import xlsxwriter
from xlsxwriter.utility import xl_rowcol_to_cell

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
from lsff_utils import config_utils

In [3]:
workbook = xlsxwriter.Workbook("results_spreadsheet.xlsx")
sheet = workbook.add_worksheet("Model Results")

In [4]:
location = "india"
vehicle = "rice"

In [5]:
daly_categories = {
    "ntd": "NTD DALYs",
    "anemia": "Anemia DALYs",
    "lbwsg": "Child YLLs",
    "maternal_disorders": "Maternal Disorders DALYs",
}

In [6]:
def averted_string(category):
    if category == "Anemia DALYs":
        # Can be due to either fortificant
        return " Averted"
    else:
        fortificant = "Folate" if category == "NTD DALYs" else "Iron"
        return f" Averted by {fortificant} Fortification"

In [7]:
header = []

for category in daly_categories.values():
    header += [
        f"Baseline {category} (1000s)",
        f"{category}{averted_string(category)} (1000s)",
        f"{category}{averted_string(category)} (%)",
    ]

header

['Baseline NTD DALYs (1000s)',
 'NTD DALYs Averted by Folate Fortification (1000s)',
 'NTD DALYs Averted by Folate Fortification (%)',
 'Baseline Anemia DALYs (1000s)',
 'Anemia DALYs Averted (1000s)',
 'Anemia DALYs Averted (%)',
 'Baseline Child YLLs (1000s)',
 'Child YLLs Averted by Iron Fortification (1000s)',
 'Child YLLs Averted by Iron Fortification (%)',
 'Baseline Maternal Disorders DALYs (1000s)',
 'Maternal Disorders DALYs Averted by Iron Fortification (1000s)',
 'Maternal Disorders DALYs Averted by Iron Fortification (%)']

In [8]:
subtotals = {
    "DALYs Averted by Fortification (1000s)": [
        "Anemia DALYs Averted (1000s)",
        "Child YLLs Averted by Iron Fortification (1000s)",
        "NTD DALYs Averted by Folate Fortification (1000s)",
    ],
}

In [9]:
header.insert(0, "DALYs Averted by Fortification (1000s)")
header

['DALYs Averted by Fortification (1000s)',
 'Baseline NTD DALYs (1000s)',
 'NTD DALYs Averted by Folate Fortification (1000s)',
 'NTD DALYs Averted by Folate Fortification (%)',
 'Baseline Anemia DALYs (1000s)',
 'Anemia DALYs Averted (1000s)',
 'Anemia DALYs Averted (%)',
 'Baseline Child YLLs (1000s)',
 'Child YLLs Averted by Iron Fortification (1000s)',
 'Child YLLs Averted by Iron Fortification (%)',
 'Baseline Maternal Disorders DALYs (1000s)',
 'Maternal Disorders DALYs Averted by Iron Fortification (1000s)',
 'Maternal Disorders DALYs Averted by Iron Fortification (%)']

In [10]:
header.append("")

In [11]:
case_categories = {
    "Annual NTD Deaths and Stillbirths": "ntd_cases_by_scenario",
    "Prevalent Anemia Cases": "prevalent_anemia_cases_by_scenario",
    "Annual Maternal Disorders Cases": "maternal_disorders_incident_cases_by_scenario",
    "Annual Child Deaths": "neonatal_deaths_by_scenario",
}

In [12]:
for category in case_categories.keys():
    header += [f"Baseline {category}", f"{category} Averted", f"{category} Averted (%)"]

header

['DALYs Averted by Fortification (1000s)',
 'Baseline NTD DALYs (1000s)',
 'NTD DALYs Averted by Folate Fortification (1000s)',
 'NTD DALYs Averted by Folate Fortification (%)',
 'Baseline Anemia DALYs (1000s)',
 'Anemia DALYs Averted (1000s)',
 'Anemia DALYs Averted (%)',
 'Baseline Child YLLs (1000s)',
 'Child YLLs Averted by Iron Fortification (1000s)',
 'Child YLLs Averted by Iron Fortification (%)',
 'Baseline Maternal Disorders DALYs (1000s)',
 'Maternal Disorders DALYs Averted by Iron Fortification (1000s)',
 'Maternal Disorders DALYs Averted by Iron Fortification (%)',
 '',
 'Baseline Annual NTD Deaths and Stillbirths',
 'Annual NTD Deaths and Stillbirths Averted',
 'Annual NTD Deaths and Stillbirths Averted (%)',
 'Baseline Prevalent Anemia Cases',
 'Prevalent Anemia Cases Averted',
 'Prevalent Anemia Cases Averted (%)',
 'Baseline Annual Maternal Disorders Cases',
 'Annual Maternal Disorders Cases Averted',
 'Annual Maternal Disorders Cases Averted (%)',
 'Baseline Annual Chi

In [13]:
num_sidebar_cols = 2
sheet.set_column(0, 0, 1)
sheet.set_column(1, 1, 10)

0

In [14]:
header_format = workbook.add_format({"bold": True, "text_wrap": True})

for idx, header_value in enumerate(header):
    sheet.write(0, idx + num_sidebar_cols, header_value, header_format)
    sheet.set_column(
        idx + num_sidebar_cols, idx + num_sidebar_cols, 15 if header_value != "" else 1
    )

In [15]:
display_quintiles = {
    1: "Poorest",
    2: "Second",
    3: "Third",
    4: "Fourth",
    5: "Wealthiest",
}

In [16]:
next_row = 1

In [17]:
comma_format = workbook.add_format(
    {"num_format": '_(* #,##0_);_(* (#,##0);_(* "-"??_);_(@_)'}
)
percent_format = workbook.add_format({"num_format": '0.0%;-0.0%;"-"'})
summary_format = workbook.add_format({"bold": True, "italic": True})
bold_format = workbook.add_format({"bold": True})

In [18]:
baseline_scenario_names = {
    "zero": "no fortification",
    "baseline": "existing and planned fortification",
}

In [19]:
intervention_scenario_names = {
    "intervention_25_nrv": "25% NRV",
    "intervention_100_nrv": "100% NRV",
}

In [20]:
import pandas as pd

In [21]:
for _, (
    location,
    vehicle,
    baseline_scenario,
    intervention_scenario,
    custom_name,
) in pd.read_csv("../0050_config/location_vehicle_scenario_comparisons.csv").iterrows():

    summary = f"{location.title()} -- {vehicle.title()}"
    if not pd.isnull(custom_name):
        summary += f"-- {custom_name}"

    sheet.write(next_row, 0, summary, summary_format)
    next_row += 1

    sheet.write(next_row, 1, "National")
    national_row = next_row
    next_row += 1

    dalys_by_scenario = pd.read_csv(
        f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
    )
    dalys_by_scenario = dalys_by_scenario.set_index(
        [c for c in dalys_by_scenario.columns if c != "value"]
    ).value

    quintile_rows = []

    sheet.write(next_row, 0, "Wealth quintile", bold_format)
    next_row += 1

    for quintile, display_quintile in display_quintiles.items():
        sheet.write(next_row, 1, display_quintile)

        for entity, header_category in daly_categories.items():
            baseline_dalys = dalys_by_scenario.loc[
                (baseline_scenario, entity, quintile)
            ]
            dalys_averted = (
                baseline_dalys
                - dalys_by_scenario.loc[(intervention_scenario, entity, quintile)]
            )

            baseline_index = (
                header.index(f"Baseline {header_category} (1000s)") + num_sidebar_cols
            )
            averted_index = (
                header.index(
                    f"{header_category}{averted_string(header_category)} (1000s)"
                )
                + num_sidebar_cols
            )

            sheet.write_number(
                next_row, baseline_index, baseline_dalys.sum() / 1_000, comma_format
            )
            sheet.write_number(
                next_row, averted_index, dalys_averted.sum() / 1_000, comma_format
            )

            formula = f"=IF({xl_rowcol_to_cell(next_row, baseline_index)}=0,0,{xl_rowcol_to_cell(next_row, averted_index)}/{xl_rowcol_to_cell(next_row, baseline_index)})"
            sheet.write_formula(
                next_row,
                header.index(f"{header_category}{averted_string(header_category)} (%)")
                + num_sidebar_cols,
                formula,
                percent_format,
            )

        for subtotal_header, sum_headers in subtotals.items():
            formula = "+".join(
                [
                    xl_rowcol_to_cell(next_row, header.index(h) + num_sidebar_cols)
                    for h in sum_headers
                ]
            )
            sheet.write_formula(
                next_row,
                header.index(subtotal_header) + num_sidebar_cols,
                formula,
                comma_format,
            )

        for header_category, case_file_name in case_categories.items():
            cases_by_scenario = pd.read_csv(
                f"./results/{location}/{vehicle}/{case_file_name}.csv"
            )
            cases_by_scenario = cases_by_scenario.set_index(
                [c for c in cases_by_scenario.columns if c != "value"]
            ).value
            baseline_cases = cases_by_scenario.loc[(baseline_scenario, quintile)]
            cases_averted = (
                baseline_cases
                - cases_by_scenario.loc[(intervention_scenario, quintile)]
            )

            baseline_index = (
                header.index(f"Baseline {header_category}") + num_sidebar_cols
            )
            averted_index = (
                header.index(f"{header_category} Averted") + num_sidebar_cols
            )

            sheet.write_number(
                next_row, baseline_index, baseline_cases.sum(), comma_format
            )
            sheet.write_number(
                next_row, averted_index, cases_averted.sum(), comma_format
            )

            formula = f"=IF({xl_rowcol_to_cell(next_row, baseline_index)}=0,0,{xl_rowcol_to_cell(next_row, averted_index)}/{xl_rowcol_to_cell(next_row, baseline_index)})"
            sheet.write_formula(
                next_row,
                header.index(f"{header_category} Averted (%)") + num_sidebar_cols,
                formula,
                percent_format,
            )

        quintile_rows.append(next_row)
        next_row += 1

    for entity, header_category in daly_categories.items():
        baseline_index = (
            header.index(f"Baseline {header_category} (1000s)") + num_sidebar_cols
        )
        averted_index = (
            header.index(f"{header_category}{averted_string(header_category)} (1000s)")
            + num_sidebar_cols
        )

        for to_sum_index in [baseline_index, averted_index]:
            formula = "+".join(
                [xl_rowcol_to_cell(row, to_sum_index) for row in quintile_rows]
            )
            sheet.write_formula(national_row, to_sum_index, formula, comma_format)

        formula = f"=IF({xl_rowcol_to_cell(national_row, baseline_index)}=0,0,{xl_rowcol_to_cell(national_row, averted_index)}/{xl_rowcol_to_cell(national_row, baseline_index)})"
        sheet.write_formula(
            national_row,
            header.index(f"{header_category}{averted_string(header_category)} (%)")
            + num_sidebar_cols,
            formula,
            percent_format,
        )

    for subtotal_header, sum_headers in subtotals.items():
        formula = "+".join(
            [
                xl_rowcol_to_cell(row, header.index(subtotal_header) + num_sidebar_cols)
                for row in quintile_rows
            ]
        )
        sheet.write_formula(
            national_row,
            header.index(subtotal_header) + num_sidebar_cols,
            formula,
            comma_format,
        )

    for header_category in case_categories.keys():
        baseline_index = header.index(f"Baseline {header_category}") + num_sidebar_cols
        averted_index = header.index(f"{header_category} Averted") + num_sidebar_cols

        for to_sum_index in [baseline_index, averted_index]:
            formula = "+".join(
                [xl_rowcol_to_cell(row, to_sum_index) for row in quintile_rows]
            )
            sheet.write_formula(national_row, to_sum_index, formula, comma_format)

        formula = f"=IF({xl_rowcol_to_cell(national_row, baseline_index)}=0,0,{xl_rowcol_to_cell(national_row, averted_index)}/{xl_rowcol_to_cell(national_row, baseline_index)})"
        sheet.write_formula(
            national_row,
            header.index(f"{header_category} Averted (%)") + num_sidebar_cols,
            formula,
            percent_format,
        )

    next_row += 1

In [22]:
# sheet.autofit()

In [23]:
sheet.freeze_panes(1, num_sidebar_cols)

In [24]:
workbook.close()